# Group Descriptive Statistics (PD vs Control)
Independent notebook: reads the CSV files already generated by `Script2.ipynb` (`ALL_patients_summary_pd.csv` and `ALL_patients_summary_control.csv`) and calculates the mean, standard deviation, and median **across patients** for each gait metric, by group.

It does not run any `skdh` code or modify raw signals -- it only works with the summaries already generated.

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import os
import pandas as pd


## 2. Configuration

In [ ]:
PD_SUMMARY_CSV      = "/content/drive/MyDrive/Lasige/PD/ALL_patients_summary_pd.csv"
CONTROL_SUMMARY_CSV = "/content/drive/MyDrive/Lasige/Control/ALL_patients_summary_control.csv"
BOUTS_VS_ACTIVITY_CSV = "/content/drive/MyDrive/Lasige/ALL_patients_bouts_vs_activity.csv"

# Patients to exclude from the analysis (e.g., known detection failures: p624, p322)
EXCLUDED_PATIENTS = ["p624", "p322"]

OUTPUT_DIR = "/content/drive/MyDrive/Lasige"
os.makedirs(OUTPUT_DIR, exist_ok=True)
OUTPUT_CSV = os.path.join(OUTPUT_DIR, "Group_descriptive_stats.csv")


Adjust the paths above if your CSV files are located elsewhere (these are the files generated by `Script2.ipynb` at the end of `process_pd()` / `process_datos_control()`).

## 3. Load and combine the PD and Control summaries

In [ ]:
pd_df = pd.read_csv(PD_SUMMARY_CSV)
control_df = pd.read_csv(CONTROL_SUMMARY_CSV)

combined = pd.concat([pd_df, control_df], ignore_index=True)
print(f"PD: {len(pd_df)} patients | Control: {len(control_df)} patients")

if EXCLUDED_PATIENTS:
    before = len(combined)
    found = combined.loc[combined["patient_id"].isin(EXCLUDED_PATIENTS), "patient_id"].tolist()
    not_found = [p for p in EXCLUDED_PATIENTS if p not in found]
    combined = combined[~combined["patient_id"].isin(EXCLUDED_PATIENTS)].reset_index(drop=True)
    print(f"Excluded: {found} ({before} -> {len(combined)} patients)")
    if not_found:
        print(f"[WARNING] not found in the data: {not_found}")

combined.head()


## 3b. Recalculate n_bouts and n_steps from ALL_patients_bouts_vs_activity.csv

Instead of using the `n_bouts`/`n_steps` already included in the summary, we recalculate them directly from the bout file (counting only those with `matched_activity == "Walking slow"`), which is the original data source.

In [ ]:
bouts_df = pd.read_csv(BOUTS_VS_ACTIVITY_CSV)

walking_bouts = bouts_df[bouts_df["matched_activity"] == "Walking slow"]

bouts_summary = (
    walking_bouts
    .groupby("patient_id")
    .agg(
        n_bouts=("bout_n", "count"),
        n_steps=("n_steps", "sum"),
        total_duration_s=("duration_s", "sum"),
    )
    .reset_index()
)

print(f"Patients with at least one 'Walking slow' bout: {len(bouts_summary)}")
bouts_summary.head()


In [ ]:
# remove the n_bouts/n_steps old del resumen and add the recalculated values
combined = combined.drop(columns=[c for c in ["n_bouts", "n_steps"] if c in combined.columns])
combined = combined.merge(bouts_summary, on="patient_id", how="left")

# patients with no Walking slow bout remain at 0, not NaN
combined[["n_bouts", "n_steps", "total_duration_s"]] = (
    combined[["n_bouts", "n_steps", "total_duration_s"]].fillna(0)
)

combined.head()


## 4. Per-patient table (to see individual records)

One row per patient, with `n_bouts`, `n_steps`, and the main metrics -- useful for identifying outliers before looking only at group averages.

In [ ]:
# key columns to display first; adjust this list if you want to view other metrics
key_metric_cols = [
    "PARAM:cadence_mean",
    "PARAM:gait speed_mean",
    "PARAM:stride time_mean",
]
key_metric_cols = [c for c in key_metric_cols if c in combined.columns]

per_patient_table = combined[
    ["patient_id", "Group", "n_bouts", "n_steps", "total_duration_s"] + key_metric_cols
].sort_values(["Group", "patient_id"]).reset_index(drop=True)

per_patient_path = os.path.join(OUTPUT_DIR, "Per_patient_table.csv")
per_patient_table.to_csv(per_patient_path, index=False)
print(f"Saved -> {per_patient_path}")

per_patient_table


If you want to see **all** columns for each patient (not only the key ones), use `combined.sort_values(["Group", "patient_id"])` directly.

## 5. Calculate descriptive statistics by metric and group

Includes `n_bouts` and `n_steps` (counts per patient) in addition to all `PARAM:*_mean` columns. We use only each patient's `_mean` columns (variability **between patients**), not `_std` (which represents variability **within each patient**, across their own gait cycles).

In [ ]:
# patient-level gait metric columns (those ending in _mean)
param_metric_cols = [c for c in combined.columns if c.startswith("PARAM:") and c.endswith("_mean")]

# + patient-level counts, which do not have the PARAM: prefix
count_cols = [c for c in ["n_bouts", "n_steps", "total_duration_s"] if c in combined.columns]

metric_cols = count_cols + param_metric_cols

print(f"{len(metric_cols)} metrics detected:")
for c in metric_cols:
    print(f"  {c}")


In [ ]:
rows = []

for col in metric_cols:
    metric_name = col.replace("PARAM:", "").replace("_mean", "")
    for group_name, group_df in combined.groupby("Group"):
        values = group_df[col].dropna()
        rows.append({
            "metric": metric_name,
            "Group": group_name,
            "n": len(values),
            "mean": values.mean(),
            "std": values.std(),
            "median": values.median(),
            "min": values.min(),
            "max": values.max(),
        })

group_stats = pd.DataFrame(rows)
group_stats = group_stats.sort_values(["metric", "Group"]).reset_index(drop=True)
group_stats


## 6. Save the result

In [ ]:
group_stats.to_csv(OUTPUT_CSV, index=False)
print(f"Saved -> {OUTPUT_CSV}")


## 7. (Optional) View a specific metric side by side

Quick example for reviewing a specific metric without having to search for it in the long table.

In [ ]:
METRIC_TO_CHECK = "cadence"  # change to any name from the list in Cell 4

group_stats[group_stats["metric"] == METRIC_TO_CHECK]


## 8. Report plots (PD vs Control boxplots)

Boxplot instead of error bars, because we already observed outliers (p322, p624...) that a mean +/- SD can hide. The boxplot shows the median, interquartile range, and marks outliers as separate points -- more informative for a report with small samples.

Adjust `REPORT_METRICS` with the metrics you want to include (use the actual column names, the same ones shown in Cells 4/12).

In [ ]:
import matplotlib.pyplot as plt

REPORT_METRICS = {
    "PARAM:cadence_mean": "Cadence (steps/min)",
    "PARAM:gait speed_mean": "Gait speed (m/s)",
    "PARAM:stride time_mean": "Stride time (s)",
    "PARAM:step length_mean": "Step length (m)",
    "PARAM:harmonic ratio - V_mean": "Harmonic ratio (V)",
    "PARAM:stride time asymmetry_mean": "Stride time asymmetry",
}
report_cols = {k: v for k, v in REPORT_METRICS.items() if k in combined.columns}
missing = [k for k in REPORT_METRICS if k not in combined.columns]
if missing:
    print(f"[WARNING] these columns don't exist in your data, skipping: {missing}")


In [ ]:
groups = sorted(combined["Group"].unique())
n = len(report_cols)
n_cols = 3
n_rows = -(-n // n_cols)  # ceiling

fig, axes = plt.subplots(n_rows, n_cols, figsize=(5*n_cols, 4*n_rows))
axes = axes.flatten() if n > 1 else [axes]

for ax, (col, label) in zip(axes, report_cols.items()):
    data = [combined.loc[combined["Group"] == g, col].dropna() for g in groups]
    ax.boxplot(data, showmeans=True)
    ax.set_xticks(range(1, len(groups) + 1))
    ax.set_xticklabels(groups)
    ax.set_title(label)
    ax.set_ylabel(label)

# turn off unused panels if the metrics do not fill the grid
for ax in axes[n:]:
    ax.axis("off")

plt.tight_layout()
boxplot_path = os.path.join(OUTPUT_DIR, "Report_boxplots.png")
plt.savefig(boxplot_path, dpi=200)
print(f"Saved -> {boxplot_path}")
plt.show()


## 9. Report table (mean +/- SD, median)

Format "mean +/- SD (median)" by metric and group, in a wide table (one row per metric, one column per group) -- ready to copy to Word/Excel or export as an image.

In [ ]:
def format_cell(row):
    return f"{row['mean']:.2f} \u00b1 {row['std']:.2f} ({row['median']:.2f})"

report_table = group_stats.copy()
report_table["formatted"] = report_table.apply(format_cell, axis=1)

report_wide = report_table.pivot(index="metric", columns="Group", values="formatted")
n_per_group = combined.groupby("Group").size()
report_wide.columns = [f"{g} (n={n_per_group[g]})" for g in report_wide.columns]

report_table_path = os.path.join(OUTPUT_DIR, "Report_table.csv")
report_wide.to_csv(report_table_path)
print(f"Saved -> {report_table_path}")

report_wide


## 10. (Optional) Export the table as an image

Useful if your report is in a format where you prefer to paste an image instead of importing a CSV (e.g., a presentation or a document where table editing is inconvenient).

In [ ]:
fig, ax = plt.subplots(figsize=(10, 0.4 * len(report_wide) + 1))
ax.axis("off")

tbl = ax.table(
    cellText=report_wide.values,
    rowLabels=report_wide.index,
    colLabels=report_wide.columns,
    cellLoc="center",
    loc="center",
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(9)
tbl.scale(1, 1.5)

plt.tight_layout()
table_img_path = os.path.join(OUTPUT_DIR, "Report_table.png")
plt.savefig(table_img_path, dpi=200, bbox_inches="tight")
print(f"Saved -> {table_img_path}")
plt.show()


## 11. Correlation heatmap

Pearson correlation between the key metrics, using all patients (PD + Control together). Useful for examining, for example, whether `gait speed` and `cadence` are strongly correlated (as expected), or whether an asymmetry metric is related to gait speed.

If you prefer to view the correlation ONLY within one group (e.g., PD only), change `CORR_GROUP` from `None` to `"PD"` or `"Control"`.

In [ ]:
import numpy as np

CORR_GROUP = None  # None = all patients together; or "PD" / "Control"

corr_source = combined if CORR_GROUP is None else combined[combined["Group"] == CORR_GROUP]
corr_cols = list(report_cols.keys())  # reuse the same metrics as the boxplots
corr_matrix = corr_source[corr_cols].corr(method="pearson")
corr_labels = [report_cols[c] for c in corr_cols]

fig, ax = plt.subplots(figsize=(1.1 * len(corr_cols) + 2, 1.1 * len(corr_cols) + 2))
im = ax.imshow(corr_matrix.values, vmin=-1, vmax=1, cmap="RdBu_r")

ax.set_xticks(range(len(corr_labels)))
ax.set_yticks(range(len(corr_labels)))
ax.set_xticklabels(corr_labels, rotation=45, ha="right")
ax.set_yticklabels(corr_labels)

for i in range(len(corr_labels)):
    for j in range(len(corr_labels)):
        ax.text(j, i, f"{corr_matrix.values[i, j]:.2f}",
                ha="center", va="center", fontsize=8,
                color="white" if abs(corr_matrix.values[i, j]) > 0.6 else "black")

title_suffix = "all patients" if CORR_GROUP is None else CORR_GROUP
ax.set_title(f"Correlation matrix ({title_suffix})")
fig.colorbar(im, ax=ax, label="Pearson r")

plt.tight_layout()
corr_path = os.path.join(OUTPUT_DIR, "Report_correlation_heatmap.png")
plt.savefig(corr_path, dpi=200, bbox_inches="tight")
print(f"Saved -> {corr_path}")
plt.show()


## 12. (Optional) Scatter plot of a specific pair

If you see a correlation of interest in the heatmap (e.g., gait speed vs. cadence), this cell plots it as a scatter plot, grouped by group.

In [ ]:
X_METRIC = "PARAM:cadence_mean"
Y_METRIC = "PARAM:gait speed_mean"

fig, ax = plt.subplots(figsize=(6, 5))
for g in groups:
    sub = combined[combined["Group"] == g]
    ax.scatter(sub[X_METRIC], sub[Y_METRIC], label=g, alpha=0.8)

r = combined[[X_METRIC, Y_METRIC]].corr().iloc[0, 1]
ax.set_xlabel(REPORT_METRICS.get(X_METRIC, X_METRIC))
ax.set_ylabel(REPORT_METRICS.get(Y_METRIC, Y_METRIC))
ax.set_title(f"r = {r:.2f}")
ax.legend()

plt.tight_layout()
scatter_path = os.path.join(OUTPUT_DIR, "Report_scatter.png")
plt.savefig(scatter_path, dpi=200, bbox_inches="tight")
print(f"Saved -> {scatter_path}")
plt.show()
